# Class 5b - Survey Data (social science)

In Class 4, each row was a **time step** measured by an instrument. Today, each row is a **person** who answered a questionnaire.

Goals:
- read survey data together with its **codebook**, and clean **missing-value codes**,
- analyse the main question types: single choice, multiple choice, **Likert scales** and **open text**,
- build **composite scores**, check their reliability, and find structure with **factor analysis**,
- compare groups with **statistical tests** and **regression**,
- use **machine learning** (scikit-learn) to explore and classify free-text answers.


> ### About this dataset
>
> The survey we are going to use is "Owning a forest – what the forest means to owners and the people around them" was run in spring 2025 by the [DIANA project](https://diana-waldstrategie.de/) at the University of Freiburg. It gathers the answers of **501 private forest owners**, most of them owning small forests (under 20 ha) in Baden-Württemberg.
>
> **What was changed:** the file you get is a modified teaching version, not the original data:
> - only the questions we use today are kept (111 of 271 columns); timestamps, durations and most "other, please specify" answers are removed,
> - everything is translated to English,
> - every respondent has a new random id,
> - small random noise was added: about 5% of the scale answers moved by one step, and forest sizes were changed by a few percent,
> - names of people and places were removed from the free text.


## 1. From sensor tables to survey tables

A survey table looks like any `DataFrame`, but the values are **codes** for answers. To read them you need the **codebook**: the list of questions, with the meaning of every code.

Question types you will meet today:

| Type | Example | Stored as |
|---|---|---|
| Single choice | age class, gender | one column, one code per person |
| Multiple choice | "How did you get your forest? (inherited / bought / ...)" | one **0/1 column per option** |
| Likert scale | "I feel emotionally connected to my forest", from *strongly agree* to *strongly disagree* | one column per statement, codes 1–5 |
| Intensity scale | "How strongly do you worry about drought?" | codes 1–5, plus 6 = *does not apply* |
| Free keywords | "Write up to 5 words that come to mind about your forest" | 5 text columns |
| Open question | "What would you do after a disaster in your forest?" | free text |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 2. Loading the data and the codebook

We read two CSV files with `pd.read_csv(...)`: the answers, and the codebook.


In [ ]:
Answers = pd.read_csv('../processed_data/diana_answers.csv').copy()
Codebook = pd.read_csv('../processed_data/diana_codebook.csv')
print(Answers.shape)
print(Codebook.shape)


What do the first columns of the answers table look like?


In [ ]:
print(Answers.columns[:15])


The column names are **question codes** (`F26`, `F10_v`, ...), as in real survey exports. The codebook tells us what they mean. Let's look up the question `F26`:


In [ ]:
print(Codebook[Codebook['variable'] == 'F26'])


## 2. Dropping useless codes

Survey software rarely writes empty cells. Instead, it uses **special codes**. In this dataset:

- `0` = *no answer* in all numeric (non-binary) questions,
- `6` = *does not apply* in the intensity scales (e.g. worries),
- `-99` = *not answered* and `-66` = *not shown* in text questions.

These codes are dangerous: if you compute the mean age class with the `0`s inside, the result is wrong.

`.replace({0: np.nan})` turns the *no answer* code into a real missing value. Pandas then skips those cells in `.mean()` and in `.value_counts()` (same idea as in Class 4: statistics ignore `NaN`).


In [ ]:
Age = Answers['F26'].replace({0: np.nan})


## 3. Calculate frequency

On a single column, **`.value_counts()`** lists every distinct value and how many people chose it (a frequency table). Use it before any mean or plot.


In [ ]:
print(Age.value_counts())


Add **`normalize=True`** to get shares instead of counts:


In [ ]:
AgeShare = Age.value_counts(True) * 100
print(AgeShare.round(1))


A good way to show that is using barplot with `plt.bar(x=..., height=...)` or `plt.barh(y=..., width=...)`:


In [ ]:
plt.bar(x=AgeShare.index, height=AgeShare)
plt.xlabel('Age class (code; see codebook)')
plt.ylabel('Share of respondents (%)')
plt.title('Age of the participants')
plt.show()


Try with `plt.barh`:


## Exercise A

Almost every survey paper starts with a "**Table 1**" describing who answered. Reviewers use it to judge how far the results can be generalised.

Tasks:
1. For **age** (`F26`), **gender** (`F27`), **education** (`F28`) and **place of residence** (`F29`), drop code `0` (*no answer*) before counting (same pattern as for age above)
2. Make one bar chart for each, as we did for age


In [ ]:
for Variable in ['F26', 'F27', 'F28', 'F29']:
    Clean = Answers[Variable].replace({0: np.nan})
    Share = Clean.value_counts(True).sort_index() * 100
    plt.bar(x=Share.index, height=Share)
    plt.xlabel('Answer code')
    plt.ylabel('Share of respondents (%)')
    plt.title(Variable)
    plt.show()


## 4. Multiple choice questions (0/1 columns)

When several answers are possible, the survey stores **one column per option**, with `1` if the option was ticked and `0` otherwise. The prefix is shared: `F1_1_a`, `F1_1_b`, ... are all options of "How did you get your forest?". Note that there are no useless values here.

Instead of `Age = Answers["F26"]` we select **several columns** with `Answers[AcquiredCols]`. Instead of `.value_counts()` (one answer code per person), we use **`.mean()`** on each 0/1 column, the mean is the share who ticked that option.


In [ ]:
AcquiredCols = ['F1_1_a', 'F1_1_b', 'F1_1_c', 'F1_1_d', 'F1_1_e']
AcquiredShare = Answers[AcquiredCols].mean() * 100


For the plot we make a list **`AcquiredLabels`** to `plt.barh` instead of `AgeShare.index` to have labels instead numeric codes.


In [ ]:
AcquiredLabels = ['I inherited the forest.', 'I bought the forest.', 'I received the forest through marriage.', 'I received the forest as a gift.', 'I received the forest in another way.']
plt.barh(y=AcquiredLabels, width=AcquiredShare)
plt.xlabel('Share of owners (%)')
plt.title('How did you get your forest?')
plt.show()


## Exercise B (Bonus)

The DIANA project asks whether the **next generation** will still care for small forests. The survey uses the same list of options twice: **who takes care of the forest today** (`F12_a`–`F12_f`) and **who will in 10 years** (`F13_a`–`F13_f`).

Tasks:
1. For each option, compute the share of owners who ticked it (same `.mean()` pattern as in section 3).
2. Draw **two** horizontal bar charts (one for today, one for in 10 years ). Set the x-axis from 0 to 100% on both (`plt.xlim(left=0, right=100)`).


In [ ]:
TodayCols = ['F12_a', 'F12_b', 'F12_c', 'F12_d', 'F12_e', 'F12_f']
TenYearsCols = ['F13_a', 'F13_b', 'F13_c', 'F13_d', 'F13_e', 'F13_f']
CareLabels = ['Nobody', 'I myself', 'Someone from my family', 'Acquaintances', 'Service providers (contractors)', 'Others']
TodayShare = Answers[TodayCols].mean() * 100
TenYearsShare = Answers[TenYearsCols].mean() * 100
plt.barh(y=CareLabels, width=TodayShare)
plt.xlim(left=0, right=100)
plt.xlabel('Share of owners (%)')
plt.title('Who takes care of the forest today?')
plt.show()
plt.barh(y=CareLabels, width=TenYearsShare)
plt.xlim(left=0, right=100)
plt.xlabel('Share of owners (%)')
plt.title('Who will take care of the forest in 10 years?')
plt.show()


## 4. Likert and intensity scales

The block `F10` contains 44 statements about the forest, each rated from `1 = strongly agree` to `5 = strongly disagree`. This is a **Likert scale**.


In [ ]:
print(Codebook[Codebook['variable'] == 'F10_v'])
print(Answers['F10_v'].value_counts())


### Diverging stacked bar chart

The recommended plot for Likert data is the **diverging stacked bar chart** ([Heiberger & Robbins, 2014](https://doi.org/10.18637/jss.v057.i05)): disagreement on the **left**, agreement on the **right**, *neutral* in the middle.

We use the **`plot-likert`** package (`import plot_likert`).


In [ ]:
import plot_likert

Items = ['F10_k', 'F10_y', 'F10_3', 'F10_v', 'F10_s', 'F10_4', 'F10_12', 'F10_z']
Agreement = Answers[Items].replace({0: np.nan})

plot_likert.plot_likert(
    df=Agreement[Items], 
    plot_scale=[5, 4, 3, 2, 1], 
    plot_percentage=True
    )
plt.title('Relationship to the forest')
plt.show()


We can change the labels with a dictionnary and the function `replace` (see next class):


In [ ]:
Scale = [
    'strongly disagree', 'disagree', 'neutral', 'agree', 'strongly agree'
]
LabelMap = {
    1: 'strongly agree',
    2: 'agree',
    3: 'neutral',
    4: 'disagree',
    5: 'strongly disagree',
}

plot_likert.plot_likert(
    df=Agreement[Items].replace(LabelMap),
    plot_scale=Scale,
    plot_percentage=True,
)
plt.title('Relationship to the forest')
plt.show()


## 5. From single items to constructs

A single survey item doesn't tell you much on its own. Researchers usually write several items that all aim at the same underlying idea, like emotional attachment or economic orientation. We call that idea a construct.

### i. Correlation heatmap

For ordinal data we use the **Spearman** correlation (it works on ranks). On a table of numbers, **`.corr(method="spearman")`** computes one correlation per column pair.


In [ ]:
Corr = Agreement.corr(method='spearman')


To plot, we can use `plt.imshow` to draw a matrix as an image:


In [ ]:
plt.imshow(Corr, cmap='RdBu_r', vmin=-1, vmax=1)

# Add colobar
plt.colorbar(label='Spearman correlation')

# Add ticks
plt.xticks(ticks=range(len(Corr)), labels=Corr.columns, rotation=90)
plt.yticks(ticks=range(len(Corr)), labels=Corr.columns)

plt.title('Correlations between the agreement statements')
plt.show()


### Cross-tabulations TODO

To see if an answer depends on **another** variable, use `pd.crosstab(index=..., columns=...)`. With `normalize="index"`, each row sums to 1, so we compare groups of different sizes fairly.

Do owners who work professionally with forests (`F30`) ask experts for advice more often (`F17_b`)?


In [ ]:
Forestry = Answers['F30'].replace({0: np.nan})
print(pd.crosstab(index=Forestry, columns=Answers['F17_b'], normalize='index').round(2))


### iii. Grouping and factor analysis

With 44 items, choosing groups by hand is hard. **Factor analysis** assumes that a few hidden factors explain the correlations between items. Each item gets a **loading** on each factor (similar to a correlation): items with high loadings on the same factor form a construct. A **varimax rotation** makes the loadings easier to read (each item loads mainly on one factor).

We use scikit-learn. Like most scikit-learn tools it works in two steps: create the model, then `.fit()` it to the data. Items are standardised first (mean 0, standard deviation 1), and rows with missing answers are dropped.


In [ ]:
from sklearn.decomposition import FactorAnalysis


For factor analysis we use every agreement item (`F10_...`), treat code `0` as missing, drop respondents with gaps, and standardise each column (mean 0, standard deviation 1):


In [ ]:
AllItems = []
for Col in Answers.columns:
    if Col.startswith('F10_'):
        AllItems.append(Col)

Agreement = Answers[AllItems].replace({0: np.nan})
Complete = Agreement.dropna()
Standardised = (Complete - Complete.mean()) / Complete.std()
print('Respondents used:', len(Complete))


Create the model and `.fit()` it on `Standardised`. Each factor’s loadings are one row of `Fa.components_` (factor 1 → `[0]`, factor 2 → `[1]`, …). The next four cells list the six strongest items on each factor:


In [ ]:
Fa = FactorAnalysis(n_components=4, rotation='varimax', random_state=0)
Fa.fit(Standardised)


Let's look to the weight of the first factor:


In [ ]:
factor = 1 
Weights = Fa.components_[factor-1].round(2)
print(Weights)


In [ ]:
Order = np.abs(Weights).argsort()[::-1][:6]
for j in Order:
    Col = Complete.columns[j]
    Question = Codebook.loc[Codebook['variable'] == Col, 'question'].iloc[0][:80]
    print(Weights[j], Question)


The **sign** of a factor is arbitrary: what matters is which items load strongly together, and whether some load in the opposite direction (e.g. "burden" against "I like working in my forest"). This is how studies of forest owners' values and objectives build their scales. The number of factors is a choice: try 3 or 5 and see if the factors stay interpretable.

## Exercise C

Tasks:
1. Build two composite scores (on the reversed `Agreement` scale):
   - `emotional_score` from `F10_g`, `F10_3`, `F10_5`, `F10_11`, `F10_1` (identity, emotional connection, memories, awe, beauty),
   - `economic_score` from `F10_s`, `F10_t`, `F10_4`, `F10_2`, `F10_n`.
2. Compute Cronbach's alpha of both scales. Are they reliable?
3. Plot the two distributions in one histogram (`plt.hist(x=..., bins=..., alpha=0.5)`). What is more important to these owners?


In [ ]:
EmotionalItems = ['F10_g', 'F10_3', 'F10_5', 'F10_11', 'F10_1']
EconomicItems = ['F10_s', 'F10_t', 'F10_4', 'F10_2', 'F10_n']
Answers['emotional_score'] = Agreement[EmotionalItems].mean(axis=1)
Answers['economic_score'] = Agreement[EconomicItems].mean(axis=1)
ItemsClean = Agreement[EmotionalItems].dropna()
K = ItemsClean.shape[1]
Alpha = K / (K - 1) * (1 - ItemsClean.var().sum() / ItemsClean.sum(axis=1).var())
print('Alpha emotional:', round(Alpha, 2))
ItemsClean = Agreement[EconomicItems].dropna()
K = ItemsClean.shape[1]
AlphaEcon = K / (K - 1) * (1 - ItemsClean.var().sum() / ItemsClean.sum(axis=1).var())
print('Alpha economic: ', round(AlphaEcon, 2))
print(Answers[['emotional_score', 'economic_score']].describe().round(2))


In [ ]:
Bins = np.arange(start=1, stop=5.25, step=0.25)
plt.hist(x=Answers['emotional_score'], bins=Bins, alpha=0.5, label='emotional')
plt.hist(x=Answers['economic_score'], bins=Bins, alpha=0.5, label='economic')
plt.xlabel('Score (1 = strongly disagree, 5 = strongly agree)')
plt.ylabel('Number of owners')
plt.legend()
plt.show()


**Short answer (example):** both scales are reliable ($\alpha \approx 0.8$). Emotional attachment is higher and less spread than economic orientation: most owners are emotionally attached, while the economic role of the forest divides them.


### i. Is the difference "real"? Mann-Whitney U test

Do owners who **inherited** their forest (`F1_1_a == 1`) care more about honouring the heritage of previous generations (`F10_v`)? Split the sample with `.loc` and compare the two groups before testing.

With a sample, a difference can appear by chance. A **statistical test** estimates how likely such a difference would be if the two groups were in truth identical (the **p-value**). For ordinal data comparing two groups, the standard test is **Mann-Whitney U** (`scipy.stats.mannwhitneyu`).

A p-value only tells you *whether* there is a difference, not *how big* it is. Always report an **effect size**. For Mann-Whitney, the **rank-biserial correlation** $r = 1 - \frac{2U}{n_1 n_2}$ goes from -1 to 1 (about 0.1 small, 0.3 medium, 0.5 large).


In [ ]:
from scipy import stats
Answers['inherited'] = Answers['F1_1_a']
Answers['heritage_agree'] = Agreement['F10_v']
Inherited = Answers.loc[Answers['inherited'] == 1, 'heritage_agree'].dropna()
NotInherited = Answers.loc[Answers['inherited'] == 0, 'heritage_agree'].dropna()
Result = stats.mannwhitneyu(x=NotInherited, y=Inherited)
EffectSize = 1 - 2 * Result.statistic / (len(Inherited) * len(NotInherited))
print('U =', round(Result.statistic), ', p =', round(Result.pvalue, 2), ', rank-biserial r =', round(EffectSize, 2))


### ii. Relationship between two ordinal variables: Spearman

Are owners who feel emotionally attached also the ones who visit their forest more often (`F34`)? Careful with the coding: in `F34`, `1 = daily` and `6 = less often`, so a **negative** correlation means *more attached = more frequent visits*.


In [ ]:
Visits = Answers['F34'].replace({0: np.nan})
Rho, P = stats.spearmanr(Answers['emotional_score'], Visits, nan_policy='omit')
print('Spearman rho =', round(Rho, 2), ', p =', round(P, 2))


### iii. Two categorical variables: chi-square test

People heard about the survey through different channels. Are owners reached via a **forestry organisation** (`F40_f`) more often in contact with a **forest owner association** (`F14_a`)? For two categorical variables, we use the **chi-square test** on the cross-table of counts.


In [ ]:
Counts = pd.crosstab(index=Answers['F40_f'], columns=Answers['F14_a'])
print(Counts)
print(pd.crosstab(index=Answers['F40_f'], columns=Answers['F14_a'], normalize='index').round(2))
Chi2, P, Dof, Expected = stats.chi2_contingency(Counts)
print('chi2 =', round(Chi2, 1), ', p =', round(P, 2))


This is a classic **sampling bias**: owners contacted through forestry networks are, not surprisingly, better networked. It is exactly why DIANA also recruited through non-forestry channels.

> **Caveats about tests**
> - A small p-value is not a large or important effect; look at effect sizes and plots.
> - If you run 44 tests, about 2 will be "significant" at p < 0.05 by pure chance (**multiple testing**). Correct (e.g. Bonferroni: divide 0.05 by the number of tests) or decide your hypotheses in advance.
> - Tests assume a random sample. Ours is not, so p-values are indications, not proofs.

### iv. Regression with statsmodels

Tests compare one variable at a time. A **regression** estimates the effect of several variables **at once**, each one "controlling" for the others. `statsmodels` lets you write the model as a formula, like in R: `outcome ~ x1 + x2`.


In [ ]:
import statsmodels.formula.api as smf
Answers['works_in_forestry'] = (Answers['F30'] == 1).astype(int)
Answers['age_class'] = Answers['F26'].replace({0: np.nan})
Model = smf.ols(formula='emotional_score ~ inherited + works_in_forestry + age_class', data=Answers).fit()
print(Model.summary().tables[1])
print('R2 =', round(Model.rsquared, 3))


Each coefficient is the change in the score when that variable increases by one, all others being equal. The table also gives a p-value and a 95% confidence interval for each coefficient.

For a **single Likert item**, social scientists often use an **ordinal logistic regression** instead of ordinary least squares. `statsmodels` provides `OrderedModel` for that; we skip the fit here to keep the notebook shorter.


A positive coefficient means higher odds of a higher category (more agreement). `np.exp(coefficient)` gives the **odds ratio**: here, inheriting the forest multiplies the odds of agreeing more with "honour the heritage" by about 3.

## Exercise D

Does the **size of the forest** explain the economic orientation of owners? The problem: the size (`F31_b`) was typed freely by respondents, so it is **text**: `"1,5"`, `"approx. 8 ha"`, `"less than 1 ha"`, `"12 hectares"`...

Tasks:
1. Use `.value_counts()` to see the different text formats.
2. Convert `F31_b` into a number of hectares in a new column `hectares`:
   - `-99` and texts without a number become `NaN`,
   - "less than 1 ha" (and similar) becomes `0.5`,
   - otherwise, use **`.str.extract(...)`** on the text column (string methods via `.str`, like vectorised text operations) to pull out the first number with a **regular expression** (look it up!), replace `,` by `.`, then `.astype(float)`.
3. Forest sizes are very skewed (a few owners have hundreds of hectares). Create `log_hectares = np.log1p(hectares)` (`log1p(x) = log(1 + x)` also works for 0).
4. Fit `economic_score ~ log_hectares + works_in_forestry + age_class + inherited`. Which variables matter? Interpret the coefficient of `log_hectares`.


In [ ]:
print(Answers['F31_b'].value_counts().head(20))
Size = Answers['F31_b'].str.lower()
Number = Size.str.extract('(\\d+(?:[.,]\\d+)?)')[0].str.replace(pat=',', repl='.').astype(float)
LessThanOne = Size.str.contains(pat='less than|under one|< ?1', regex=True)
Answers['hectares'] = Number
Answers.loc[LessThanOne, 'hectares'] = 0.5
Answers.loc[Answers['F31_b'].isin(['-99', '-66']), 'hectares'] = np.nan
print(Answers['hectares'].describe().round(1))


In [ ]:
Answers['log_hectares'] = np.log1p(Answers['hectares'])
plt.hist(x=Answers['log_hectares'].dropna(), bins=30)
plt.xlabel('log(1 + hectares)')
plt.ylabel('Number of owners')
plt.show()
EconomicModel = smf.ols(formula='economic_score ~ log_hectares + works_in_forestry + age_class + inherited', data=Answers).fit()
print(EconomicModel.summary().tables[1])
print('R2 =', round(EconomicModel.rsquared, 3), ' n =', int(EconomicModel.nobs))


**4. Short answer (example):** owners of **larger forests** and owners who **work in forestry** are clearly more economically oriented; older owners slightly less. Inheritance does not matter once the other variables are included. Since size is on a log scale, doubling the forest area raises the economic score by about $0.24 \times \log(2) \approx 0.17$ points. Together these variables explain about a quarter of the differences between owners ($R^2 \approx 0.25$).


## 7. Text as data: machine learning on open answers

Open answers are rich but hard to count. The traditional method is **qualitative coding**: researchers read every answer and assign categories, ideally two coders independently, then check their agreement (**inter-coder reliability**). This is slow for hundreds of answers.

Machine learning can help in two ways (see Grimmer & Stewart, 2013, *Text as Data*):
- **unsupervised**: let an algorithm find recurring **topics**, then read examples to interpret them,
- **supervised**: code a **sample** by hand, train a model on it, and let it code the rest.

In both cases, the computer never replaces reading: you **validate** the results by going back to the answers.

> Open answers in this teaching file are in **English** (translated from the original German survey). The methods are the same in any language; for full sentences you typically drop **stop words** — very frequent words like "the", "and", "would" — with `stop_words="english"` in scikit-learn.

### i. Bag of words

Models cannot read text directly: we turn each answer (a **document**) into a row of word counts, the **document-term matrix**. Word order is lost, hence the name *bag of words*. In scikit-learn this is `CountVectorizer`.

Our first documents are the **free associations** (`F4_a` to `F4_e`): each owner wrote up to five words that come to mind about their forest. We join them into one document per owner, linking multi-word expressions with `_` (e.g. `peace_and_quiet`) so that they stay one token.

We build one text per owner with **`.apply(..., axis=1)`** (run a function on each row). **`.fillna("")`** turns empty cells into blank strings.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
KeywordCols = ['F4_a', 'F4_b', 'F4_c', 'F4_d', 'F4_e']
Keywords = Answers[KeywordCols].replace({'-99': '', '-66': ''}).fillna('')

def merge_keywords(Row):
    Words = []
    for w in Row:
        if w:
            Words.append(w.replace(' ', '_'))
    return ' '.join(Words)
KeywordDocs = Keywords.apply(func=merge_keywords, axis=1)
print(KeywordDocs.head())
Counter = CountVectorizer(token_pattern='[^ ]+')
Dtm = Counter.fit_transform(KeywordDocs)
print('Document-term matrix:', Dtm.shape)
WordCounts = pd.Series(data=np.asarray(Dtm.sum(0)).ravel(), index=Counter.get_feature_names_out())
Top20 = WordCounts.sort_values().tail(20)
plt.barh(y=Top20.index, width=Top20)
plt.xlabel('Number of owners mentioning the word')
plt.title("Spontaneous associations with 'my forest'")
plt.show()


*work*, *recreation*, *nature*, *bark_beetle* and *firewood* lead the list: the forest is at once a workplace, a place to relax, nature, a threat and a resource.

### ii. Topic modelling with TF-IDF and NMF

Counting words does not tell us which words **go together**. A **topic model** finds groups of words that often appear in the same documents.

1. **TF-IDF** (`TfidfVectorizer`) re-weights the counts: words used by everybody get a low weight, words typical of some documents a high weight. `min_df=3` ignores words used by fewer than 3 owners.
2. **NMF** (non-negative matrix factorisation) decomposes the matrix into $k$ topics. Each topic is a list of word weights; each owner gets a weight on each topic.

The number of topics $k$ is **your choice**: try several and keep the most interpretable one.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
Tfidf = TfidfVectorizer(token_pattern='[^ ]+', min_df=3)
KeywordTfidf = Tfidf.fit_transform(KeywordDocs)
Words = Tfidf.get_feature_names_out()
KeywordNmf = NMF(n_components=4, init='nndsvda', random_state=0, max_iter=500)
KeywordTopics = KeywordNmf.fit_transform(KeywordTfidf)
for i in range(KeywordNmf.components_.shape[0]):
    Weights = KeywordNmf.components_[i]
    Order = Weights.argsort()[::-1][:8]
    TopWords = []
    for j in Order:
        TopWords.append(Words[j])
    print('Topic', i, ':', ', '.join(TopWords))


The topics can be named, for example: **work and wood** (`work`, `firewood`, `hobby`), **nature** (`nature`, `animals`, `peace_and_quiet`), **threats and forest conversion** (`bark_beetle`, `drought`, `climate_change`, `forest_conversion`) and **sustainability and recreation** (`recreation`, `sustainability`, `nature_conservation`).

The topic weights are new **numeric variables** for each owner, so we can connect the text to the rest of the survey. Do owners whose associations are about **threats** also report stronger worries about drought, storm and bark beetle?


In [ ]:
ThreatTopic = 2
WorryCols = ['F15_a', 'F15_b', 'F15_e']
Worries = Answers[WorryCols].replace({0: np.nan, 6: np.nan})
ClimateWorry = Worries.mean(axis=1)
Rho, P = stats.spearmanr(a=KeywordTopics[:, ThreatTopic], b=ClimateWorry, nan_policy='omit')
print('Spearman rho =', round(Rho, 2), ', p =', round(P, 2))


### iii. Topics in open answers (disaster scenario)

`F19` asked: *"What would you do if a disaster killed half of the trees in your forest and the other half were threatened?"*. These are full sentences, so we remove **stop words** (words that carry no topic). scikit-learn has a built-in English list (`stop_words="english"`).

Skip missing text answers with **`~...isin(["-99", "-66"])`**: `isin` tests membership, `~` negates (keep rows that are *not* missing codes).


In [ ]:
print(Codebook.loc[Codebook['variable'] == 'F19', 'question'].iloc[0])
Disaster = Answers.loc[~Answers['F19'].isin(['-99', '-66']), 'F19']
print('Answers:', len(Disaster))
DisasterTfidf = TfidfVectorizer(stop_words='english', min_df=5)
DisasterMatrix = DisasterTfidf.fit_transform(Disaster)
DisasterWords = DisasterTfidf.get_feature_names_out()
DisasterNmf = NMF(n_components=4, init='nndsvda', random_state=0, max_iter=500)
DisasterTopics = DisasterNmf.fit_transform(DisasterMatrix)
for i in range(DisasterNmf.components_.shape[0]):
    Weights = DisasterNmf.components_[i]
    Order = Weights.argsort()[::-1][:8]
    TopWords = []
    for j in Order:
        TopWords.append(DisasterWords[j])
    print('\nTopic', i, ':', ', '.join(TopWords))
    TopRows = DisasterTopics[:, i].argsort()[::-1][:3]
    for Row in TopRows:
        print('   -', Disaster.iloc[Row][:120])


Reading the typical answers is the **validation** step: do the answers really share an idea? Here we find processing and selling the wood (`process`, `sell`, `firewood`), clear-cutting and replanting (`clear-cut`, `replant`), and reforesting with suitable, climate-adapted species (`species`, `reforest`, `climate`). If a topic mixes unrelated answers, change $k$, `min_df` or the stop words.

## Exercise E

`F25` asked who would decide if the forest were **sold**.

Tasks:
1. Build the same supervised pipeline for `F25` with the labels of `diana_text_labels.csv` (`question == "F25"`). Look at the label counts first: a category with only one or two examples cannot be learned (and `stratify` fails). Merge rare categories into `other` before splitting.
2. Evaluate the model on a test set.
3. Predict the category of all `F25` answers and compare with the disaster scenario (`decides_disaster`) using `pd.crosstab`. Do the same people decide in both situations?


In [ ]:
CodedSale = TextLabels[TextLabels['question'] == 'F25'].merge(Answers[['respondent_id', 'F25']], on='respondent_id')
print(CodedSale['label'].value_counts())
CodedSale['label'] = CodedSale['label'].replace({'forester_or_expert': 'other'})
SaleTrain, SaleTest, SaleLabelTrain, SaleLabelTest = train_test_split(CodedSale['F25'], CodedSale['label'], test_size=0.3, random_state=0, stratify=CodedSale['label'])
SaleVectorizer = TfidfVectorizer(ngram_range=(1, 2))
SaleClassifier = LogisticRegression(max_iter=1000, class_weight='balanced')
SaleClassifier.fit(SaleVectorizer.fit_transform(SaleTrain), SaleLabelTrain)
print(classification_report(y_true=SaleLabelTest, y_pred=SaleClassifier.predict(SaleVectorizer.transform(SaleTest)), zero_division=0))


In [ ]:
AnsweredSale = ~Answers['F25'].isin(['-99', '-66'])
Answers.loc[AnsweredSale, 'decides_sale'] = SaleClassifier.predict(SaleVectorizer.transform(Answers.loc[AnsweredSale, 'F25']))
print(pd.crosstab(index=Answers['decides_disaster'], columns=Answers['decides_sale']))


**3. Short answer (example):** most owners decide **alone** in both scenarios, and owners who involve their **family** in one situation usually do so in the other. Experts appear in the disaster scenario (a technical question) but almost never for a sale (a family and property question).


## Exercise F (Bonus): a typology of forest owners

A classic result of forest-owner research is a **typology**: groups of owners with similar objectives, used to design advice and policy (e.g. [Tiebel et al., 2024](https://doi.org/10.1002/pan3.10579) found *multiple-use*, *conventional* and *conservation-oriented* owners in Lower Saxony). They are built with **cluster analysis**.

Tasks:
1. Build (or reuse) five composite scores from the factor analysis: `emotional_score`, `economic_score`, `future_score`, plus an **engagement** score (`F10_b`, `F10_8`, `F10_i`, `F10_5`) and an **ecological** score (`F10_l`, `F10_r`, `F10_m`, `F10_0`, `F10_k`).
2. Standardise them (`sklearn.preprocessing.StandardScaler`) and draw a **dendrogram** with `scipy.cluster.hierarchy.linkage(..., method="ward")` and `dendrogram(...)`. How many clusters would you choose?
3. Run `AgglomerativeClustering(n_clusters=k, linkage="ward")` and describe each cluster (mean of each score per `owner_type`, using `.loc` or a loop over cluster labels). Add the share of owners who work in forestry and the mean weight of each keyword topic (`pd.concat(..., axis=1)` pastes columns side by side).
4. Check with a **Kruskal-Wallis** test (`stats.kruskal`, the extension of Mann-Whitney to more than two groups) that the clusters differ in their economic score.
5. Give each type a name.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import linkage, dendrogram
Answers['engagement_score'] = Agreement[['F10_b', 'F10_8', 'F10_i', 'F10_5']].mean(axis=1)
Answers['ecological_score'] = Agreement[['F10_l', 'F10_r', 'F10_m', 'F10_0', 'F10_k']].mean(axis=1)
ScoreCols = ['emotional_score', 'economic_score', 'future_score', 'engagement_score', 'ecological_score']
Scores = Answers[ScoreCols].dropna()
ScoresStd = StandardScaler().fit_transform(Scores)
plt.figure(figsize=(10, 4))
dendrogram(Z=linkage(y=ScoresStd, method='ward'), no_labels=True)
plt.ylabel('Ward distance')
plt.title('Dendrogram of forest owners')
plt.show()


In [ ]:
Clusters = AgglomerativeClustering(n_clusters=3, linkage='ward').fit_predict(ScoresStd)
Answers.loc[Scores.index, 'owner_type'] = Clusters
Topics = pd.DataFrame(data=KeywordTopics, columns=['topic_0', 'topic_1', 'topic_2', 'topic_3'])
Profile = pd.concat(objs=[Answers[ScoreCols + ['works_in_forestry', 'owner_type']], Topics], axis=1)
print(Answers['owner_type'].value_counts())
for Cluster in sorted(Profile['owner_type'].dropna().unique()):
    print(f'\nCluster {int(Cluster)}')
    print(Profile.loc[Profile['owner_type'] == Cluster].mean(True).round(2))
Group0 = Answers.loc[Answers['owner_type'] == 0, 'economic_score']
Group1 = Answers.loc[Answers['owner_type'] == 1, 'economic_score']
Group2 = Answers.loc[Answers['owner_type'] == 2, 'economic_score']
print(stats.kruskal(Group0, Group1, Group2))


**5. Short answer (example):** with 3 clusters we find
- **multifunctional owners**: high on every dimension, including economic use,
- **attached, non-economic owners**: strong emotional and ecological values, low economic orientation,
- **distant owners**: lower scores everywhere, the forest plays a smaller role in their life.

This resembles published typologies, but the number and names of clusters are **interpretations**: another $k$, other scores or another method can give other types. Researchers justify their choice with the dendrogram, with stability checks, and with what makes sense for the research question.
